# 01 – Exploration du dataset (Gomez et al., Zenodo)

**Avant de commencer :** Exécution → Modifier le type d'exécution → GPU.

Ce notebook : monte Drive, télécharge les annotations (53 ko), liste le contenu de l'archive vidéo **sans la télécharger en entier**, puis explore les phases.

In [ ]:
import torch
print('GPU disponible :', torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
BASE = '/content/drive/MyDrive/embryo_project'
for d in ['annotations', 'videos_echantillon', 'models']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
print('Dossiers prêts dans', BASE)

## 1. Annotations (petit fichier)

In [ ]:
URL_ANN = 'https://zenodo.org/records/6390798/files/embryo_dataset_annotations.tar.gz?download=1'
!wget -q -O {BASE}/annotations/ann.tar.gz "{URL_ANN}"
!tar -xzf {BASE}/annotations/ann.tar.gz -C {BASE}/annotations
!find {BASE}/annotations | head -20

## 2. Contenu de l'archive vidéo (streaming, sans tout télécharger)

La commande s'arrête d'elle-même après 50 lignes. Un message d'erreur de type `curl: (23)` à la fin est normal. **Colle-moi le résultat.**

In [ ]:
URL_VID = 'https://zenodo.org/records/6390798/files/embryo_dataset.tar.gz?download=1'
!curl -sL "{URL_VID}" | tar -tz 2>/dev/null | head -50

## 3. Explorer les phases

In [ ]:
import glob, pandas as pd

fichiers = glob.glob(f'{BASE}/annotations/**/*.csv', recursive=True)
print(len(fichiers), 'fichiers CSV')

lignes = []
for f in fichiers:
    d = pd.read_csv(f, header=None, names=['phase', 'debut', 'fin'])
    d['debut'] = pd.to_numeric(d['debut'], errors='coerce')
    d['fin'] = pd.to_numeric(d['fin'], errors='coerce')
    d = d.dropna(subset=['debut', 'fin'])
    d['video'] = os.path.basename(f)
    lignes.append(d)

ann = pd.concat(lignes, ignore_index=True)
ann['nb_images'] = ann['fin'] - ann['debut'] + 1
print(ann.groupby('phase')['nb_images'].agg(['count', 'sum']).sort_values('sum', ascending=False))
ann.head()

In [ ]:
# Vidéos les plus complètes (le plus de phases différentes)
completes = ann.groupby('video')['phase'].nunique().sort_values(ascending=False)
completes.head(30)